# MongoDB workload generation

A pattern fixes fields, operators and query options; its instances vary filter values. Start offline with a toy schema, then optionally check selectivity using the Docker MongoDB demo. All code is synchronous and works in Jupyter.

In [ ]:
from collections import Counter
from pathlib import Path
from pprint import pprint
from tempfile import TemporaryDirectory

from dbrtoolkit.mongodb import (
    MongoCollectionRef,
    MongoExecutor,
    MongoFindQuery,
    MongoGenerationConfig,
    MongoQueryPattern,
    MongoSampleEstimator,
    MongoWorkloadGenerator,
)
from dbrtoolkit.schema import Schema
from dbrtoolkit.workload import Workload

documents = [
    {"city": city, "stars": stars, "reviews": reviews}
    for city in ["Sydney", "Melbourne", "Brisbane"]
    for stars in [2.5, 3.0, 4.0, 4.5, 5.0]
    for reviews in [10, 50, 100]
]
schema = Schema.from_documents(documents, values="full")
target = MongoCollectionRef("demo", "business")
config = MongoGenerationConfig(min_predicates=1, max_predicates=2)
generator = MongoWorkloadGenerator(target=target, schema=schema, config=config)

## Ten patterns, fifty instances each
Full-retention schemas work too: frequencies are ignored, and values are sampled uniformly from distinct observed values. Returned pattern IDs index the pattern list.

In [ ]:
workload, patterns = generator.generate(
    num_patterns=10,
    instances_per_pattern=50,
    seed=42,
    return_patterns=True,
)
assert len(workload) == 500
print(Counter(entry.pattern_id for entry in workload))
entry = workload[0]
pprint(patterns[entry.pattern_id])
pprint(entry.query.filter)
assert all(patterns[e.pattern_id].matches(e.query) for e in workload)
pprint(dict(generator.last_stats))

## Save, load and repeat
JSONL preserves queries and pattern IDs. Pattern shapes can be recovered from queries. Set instances_per_pattern=1 for one query per unique pattern.

In [ ]:
with TemporaryDirectory() as folder:
    path = Path(folder) / "workload.jsonl"
    workload.save(path)
    restored = Workload.load(path)
    assert [(e.query.fingerprint(), e.pattern_id) for e in restored] == [
        (e.query.fingerprint(), e.pattern_id) for e in workload
    ]

repeated = generator.generate(num_patterns=10, instances_per_pattern=50, seed=42)
assert [e.query.fingerprint() for e in repeated] == [e.query.fingerprint() for e in workload]
recovered_pattern = MongoQueryPattern.from_query(restored[0].query)
print(recovered_pattern == patterns[restored[0].pattern_id])

## Optional: selectivity constraints against Docker MongoDB
Start and initialize the local MongoDB Docker environment first. This cell reads demo.business, creates an owned temporary sample, and removes it on exit. It never modifies source documents. A four-document sample illustrates the API; use a representative larger sample for research. Set RUN_LIVE=True to run.

In [ ]:
RUN_LIVE = False

if RUN_LIVE:
    with MongoExecutor.from_uri("mongodb://localhost:27017", timeout_seconds=10) as executor:
        documents = executor.run(
            MongoFindQuery(target, {}, limit=100), collect_results=True
        ).documents
        live_schema = Schema.from_documents(documents, values="distinct")
        live_config = MongoGenerationConfig(
            min_predicates=1,
            max_predicates=2,
            include_fields=["city", "stars", "review_count"],
        )
        with MongoSampleEstimator(executor, target, sample_size=4, persist=True) as estimator:
            constrained = MongoWorkloadGenerator(
                target=target,
                schema=live_schema,
                config=live_config,
                estimator=estimator,
            )
            selected, selected_patterns = constrained.generate(
                num_patterns=10,
                instances_per_pattern=50,
                min_selectivity=0.25,
                max_selectivity=0.75,
                seed=42,
                return_patterns=True,
            )
            assert len(selected) == 500
            assert all(0.25 <= estimator.estimate(e.query).selectivity <= 0.75 for e in selected)
            pprint(dict(constrained.last_stats))
        assert estimator.sample_target is None

The bounds apply to every instance before skip/limit, relative to the estimator sample. Zero sample matches do not prove zero population matches. Database sampling is not controlled by the generator seed. A failed generation raises WorkloadGenerationError with diagnostics instead of returning an incomplete workload.

For custom constraints, pass validate_pattern(candidate, accepted_patterns). More operator, sort, projection and field controls are documented in the workload generation guide. Logical expressions and aggregation generation are deferred.